# Advanced 04 — Embodied Vision & Vision-Language-Action Models

## From visual grounding to closed-loop action

This credential-free lab uses a synthetic 2D workcell to expose the complete embodied contract:

```text
observe → ground → afford → propose → validate → permit → simulate → verify → recover/replan
```

The default path is **simulation only**. `LocalGroundingProxy` is not a VLM. `LocalVLAPolicyProxy` is not a foundation model or production robot policy. No cell connects to hardware, a robot SDK, ROS, a remote service, or an actuator. Policy output is untrusted data, and every exported artifact states that physical authorization is absent.


## 1. Reproducible runtime and evidence directory

All policy, normalization, freshness, and constraint choices are made on Site A training or Site B development data. Site C is reporting-only. The artifact directory is local to the notebook copy used by validation.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, replace
from hashlib import sha256
from pathlib import Path
from typing import Literal
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

SEED = 20260917
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 120)

COURSE_DIR = Path.cwd()
ARTIFACT_DIR = COURSE_DIR / ".artifacts" / "advanced-04-embodied-vla"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this synthetic notebook runtime only."
print({"python": platform.python_version(), "seed": SEED, "boundary": "local simulation only"})


## 2. Typed embodiment, observation, scene, goal, and action contracts

The dataclasses are deliberately strict. An action without a frame, physical units, control mode, observation timestamp, and embodiment version is invalid. Position arrays are two-dimensional only because the teaching simulator is 2D; the production contract would include 3D translation and orientation.


In [ ]:
@dataclass(frozen=True)
class EmbodimentContract:
    robot_id: str
    contract_version: str
    joints: int
    control_mode: Literal["cartesian_delta"]
    action_frame: Literal["robot_base"]
    workspace_x_m: tuple[float, float]
    workspace_y_m: tuple[float, float]
    max_translation_delta_m: float
    gripper_type: Literal["parallel"]
    gripper_max_width_m: float
    control_frequency_hz: float

    def __post_init__(self) -> None:
        assert self.robot_id and self.contract_version
        assert self.joints > 0 and self.max_translation_delta_m > 0
        assert self.workspace_x_m[0] < self.workspace_x_m[1]
        assert self.workspace_y_m[0] < self.workspace_y_m[1]
        assert self.gripper_max_width_m > 0 and self.control_frequency_hz > 0


@dataclass(frozen=True)
class SceneObject:
    object_id: str
    kind: Literal["block", "tray", "obstacle"]
    color: str
    position_m: tuple[float, float]
    frame: Literal["camera_overhead", "robot_base"]
    width_m: float
    visible: bool = True


@dataclass(frozen=True)
class Observation:
    observation_id: str
    timestamp_s: float
    camera_captured_at_s: float
    camera_frame: Literal["camera_overhead"]
    end_effector_position_m: tuple[float, float]
    end_effector_frame: Literal["robot_base"]
    joint_positions_rad: tuple[float, ...]
    gripper_width_m: float
    embodiment_version: str
    objects: tuple[SceneObject, ...]
    source_id: str

    def __post_init__(self) -> None:
        assert self.timestamp_s >= self.camera_captured_at_s
        assert len(self.joint_positions_rad) > 0
        assert self.gripper_width_m >= 0


@dataclass(frozen=True)
class Goal:
    instruction: str
    target_color: str
    target_kind: str
    destination_color: str
    destination_kind: str


@dataclass(frozen=True)
class GroundedGoal:
    state: Literal["grounded", "clarification_required", "review_required"]
    target_object_id: str | None
    destination_object_id: str | None
    evidence_object_ids: tuple[str, ...]
    reason: str


@dataclass(frozen=True)
class ActionProposal:
    action_id: str
    observation_id: str
    observation_timestamp_s: float
    source_capture_timestamp_s: float
    generated_at_s: float
    robot_id: str
    embodiment_version: str
    control_mode: Literal["cartesian_delta"]
    frame: Literal["robot_base"]
    translation_unit: Literal["metre"]
    translation_delta_m: tuple[float, float]
    gripper_command: Literal["open", "close", "hold"]
    skill: Literal["move_to", "grasp", "release"]
    target_object_id: str | None
    policy_version: str
    policy_confidence: float

    def __post_init__(self) -> None:
        assert self.frame == "robot_base", "Action without frame is invalid"
        assert self.translation_unit == "metre"
        assert self.control_mode == "cartesian_delta"
        assert len(self.translation_delta_m) == 2
        assert 0 <= self.policy_confidence <= 1
        assert self.source_capture_timestamp_s <= self.observation_timestamp_s <= self.generated_at_s


ARM_A = EmbodimentContract(
    robot_id="arm_A",
    contract_version="arm_A/cartesian_delta/v1",
    joints=6,
    control_mode="cartesian_delta",
    action_frame="robot_base",
    workspace_x_m=(-0.60, 0.60),
    workspace_y_m=(-0.50, 0.50),
    max_translation_delta_m=0.08,
    gripper_type="parallel",
    gripper_max_width_m=0.08,
    control_frequency_hz=10.0,
)

ARM_B = replace(
    ARM_A,
    robot_id="arm_B",
    contract_version="arm_B/cartesian_delta/v1",
    workspace_x_m=(-0.42, 0.42),
    gripper_max_width_m=0.05,
)

print(json.dumps(asdict(ARM_A), indent=2))


## 3. Coordinate transforms and assertion-tested action semantics

The overhead camera has a known teaching transform into the robot-base frame. The transform is a calibrated data dependency, not a learned policy output. We test direction and inverse consistency. We also prove that missing or wrong action frames are rejected.


In [ ]:
CAMERA_TO_BASE = np.array([
    [0.0, -1.0, 0.40],
    [1.0,  0.0, -0.20],
    [0.0,  0.0, 1.0],
])
BASE_TO_CAMERA = np.linalg.inv(CAMERA_TO_BASE)

def transform_point(point_xy: tuple[float, float], matrix: np.ndarray) -> tuple[float, float]:
    value = matrix @ np.array([point_xy[0], point_xy[1], 1.0])
    return float(value[0]), float(value[1])

point_camera = (0.20, 0.10)
point_base = transform_point(point_camera, CAMERA_TO_BASE)
round_trip = transform_point(point_base, BASE_TO_CAMERA)
assert np.allclose(round_trip, point_camera)

try:
    ActionProposal(
        "bad-frame", "obs-0", 0.0, 0.0, 0.1, "arm_A", ARM_A.contract_version,
        "cartesian_delta", "", "metre", (0.01, 0.0), "hold", "move_to", None, "proxy/v1", 0.99,
    )
    raise AssertionError("Missing frame was not rejected")
except AssertionError as exc:
    assert "frame" in str(exc).lower()

print({"camera_point": point_camera, "base_point": point_base, "round_trip": round_trip, "missing_frame": "rejected"})


## 4. Goal grounding: unique binding versus clarification

`LocalGroundingProxy` is a controlled parser over known goal fields and observed entity attributes. It is **not a VLM, foundation model, or grounding benchmark**. Evaluation-only truth is never passed into the grounding function.


In [ ]:
class LocalGroundingProxy:
    engine = "local_grounding_proxy"
    foundation_model = False

    @staticmethod
    def ground(goal: Goal, observation: Observation) -> GroundedGoal:
        visible = [item for item in observation.objects if item.visible]
        targets = [item for item in visible if item.color == goal.target_color and item.kind == goal.target_kind]
        destinations = [item for item in visible if item.color == goal.destination_color and item.kind == goal.destination_kind]
        evidence = tuple(item.object_id for item in targets + destinations)
        if len(targets) != 1:
            return GroundedGoal("clarification_required", None, None, evidence, f"target_candidates={len(targets)}")
        if len(destinations) != 1:
            return GroundedGoal("clarification_required", None, None, evidence, f"destination_candidates={len(destinations)}")
        return GroundedGoal("grounded", targets[0].object_id, destinations[0].object_id, evidence, "unique visible bindings")


def make_observation(objects: list[SceneObject], source_id: str = "Site A", now_s: float = 10.0,
                     ee: tuple[float, float] = (-0.20, -0.20), gripper_width_m: float = 0.08,
                     embodiment: EmbodimentContract = ARM_A) -> Observation:
    return Observation(
        observation_id=f"obs-{source_id.replace(' ', '-')}-{int(now_s * 10)}",
        timestamp_s=now_s,
        camera_captured_at_s=now_s - 0.02,
        camera_frame="camera_overhead",
        end_effector_position_m=ee,
        end_effector_frame="robot_base",
        joint_positions_rad=(0.0,) * embodiment.joints,
        gripper_width_m=gripper_width_m,
        embodiment_version=embodiment.contract_version,
        objects=tuple(objects),
        source_id=source_id,
    )


GOAL = Goal("Put the red block in the blue tray", "red", "block", "blue", "tray")
unique_objects = [
    SceneObject("red_block_1", "block", "red", (0.12, 0.12), "robot_base", 0.04),
    SceneObject("blue_tray_1", "tray", "blue", (0.36, 0.24), "robot_base", 0.16),
    SceneObject("restricted_1", "obstacle", "gray", (0.05, 0.02), "robot_base", 0.12),
]
ambiguous_objects = unique_objects + [SceneObject("red_block_2", "block", "red", (-0.05, 0.22), "robot_base", 0.04)]

unique_grounding = LocalGroundingProxy.ground(GOAL, make_observation(unique_objects))
ambiguous_grounding = LocalGroundingProxy.ground(GOAL, make_observation(ambiguous_objects))
assert unique_grounding.state == "grounded"
assert ambiguous_grounding.state == "clarification_required"

grounding_cases = pd.DataFrame([
    {"case": "unique", "expected": "grounded", "actual": unique_grounding.state, "action_allowed": unique_grounding.state == "grounded"},
    {"case": "ambiguous", "expected": "clarification_required", "actual": ambiguous_grounding.state, "action_allowed": ambiguous_grounding.state == "grounded"},
])
grounding_cases


The notebook's ambiguity policy is structural: an unresolved physical referent blocks action. The local proxy's confidence cannot override candidate count. In production, detection misses and identity uncertainty would require additional unknown/review states rather than converting absence into certainty.


## 5. Embodiment-conditioned affordances

The grasp proxy checks object width plus clearance against the current tool. It is intentionally incomplete: compatibility does not prove reachability, collision freedom, contact stability, payload capacity, or successful grasp.


In [ ]:
@dataclass(frozen=True)
class AffordanceRecord:
    object_id: str
    affordance: Literal["top_grasp"]
    embodiment_version: str
    region_center_m: tuple[float, float]
    region_frame: Literal["robot_base"]
    compatible: bool
    reason: str


def grasp_affordance(obj: SceneObject, embodiment: EmbodimentContract, clearance_m: float = 0.01) -> AffordanceRecord:
    required_width = obj.width_m + clearance_m
    compatible = obj.kind == "block" and required_width <= embodiment.gripper_max_width_m
    reason = "width_with_margin_within_gripper" if compatible else "tool_or_width_incompatible"
    return AffordanceRecord(obj.object_id, "top_grasp", embodiment.contract_version, obj.position_m, "robot_base", compatible, reason)


wide_block = SceneObject("wide_block", "block", "red", (0.10, 0.10), "robot_base", 0.06)
affordance_table = pd.DataFrame([
    asdict(grasp_affordance(wide_block, ARM_A)),
    asdict(grasp_affordance(wide_block, ARM_B)),
])
assert affordance_table["compatible"].tolist() == [True, False]
affordance_table


## 6. Action normalization and tokenization

The model-facing vector is `[Δx_m, Δy_m, gripper]`. The raw physical representation, normalized value, inverse transformation, and conversion version remain together. Quantization error is reported in metres rather than hidden behind token accuracy.


In [ ]:
@dataclass(frozen=True)
class ActionNormalizer:
    version: str
    low: tuple[float, float, float]
    high: tuple[float, float, float]

    def normalize(self, raw: np.ndarray) -> np.ndarray:
        low, high = np.asarray(self.low), np.asarray(self.high)
        assert np.all(raw >= low) and np.all(raw <= high), "Raw action outside normalization contract"
        return 2.0 * (raw - low) / (high - low) - 1.0

    def denormalize(self, normalized: np.ndarray) -> np.ndarray:
        low, high = np.asarray(self.low), np.asarray(self.high)
        assert np.all(normalized >= -1.0 - 1e-9) and np.all(normalized <= 1.0 + 1e-9)
        return low + (normalized + 1.0) * 0.5 * (high - low)


NORMALIZER = ActionNormalizer("cartesian-delta-2d-gripper/v1", (-0.08, -0.08, 0.0), (0.08, 0.08, 1.0))

def tokenize_action(raw: np.ndarray, bins: int = 31) -> tuple[np.ndarray, np.ndarray]:
    normalized = NORMALIZER.normalize(raw)
    tokens = np.rint((normalized + 1.0) * 0.5 * (bins - 1)).astype(int)
    reconstructed_normalized = 2.0 * tokens / (bins - 1) - 1.0
    return tokens, NORMALIZER.denormalize(reconstructed_normalized)


raw_action = np.array([0.033, -0.017, 1.0])
normalized_action = NORMALIZER.normalize(raw_action)
round_trip_action = NORMALIZER.denormalize(normalized_action)
action_tokens, reconstructed_action = tokenize_action(raw_action)
tokenization_translation_vector_error_m = float(np.linalg.norm(reconstructed_action[:2] - raw_action[:2]))
assert np.allclose(raw_action, round_trip_action)

pd.DataFrame({
    "channel": ["delta_x_m", "delta_y_m", "gripper"],
    "raw": raw_action,
    "normalized": normalized_action,
    "token": action_tokens,
    "reconstructed": reconstructed_action,
})


## 7. Behavioral cloning and the proprioception ablation

Site A demonstrations pair a uniquely grounded object position, the end-effector state, gripper state, and expert delta. The vision-only baseline sees the object and tray but not the robot configuration. The proprioceptive model sees the complete synthetic state. Both use the same Ridge API, split, and physical-unit metric.


In [ ]:
def expert_delta(ee: np.ndarray, target: np.ndarray, max_step_m: float = 0.08) -> np.ndarray:
    displacement = target - ee
    norm = np.linalg.norm(displacement)
    if norm <= max_step_m:
        return displacement
    return displacement / norm * max_step_m


def build_demonstrations(n: int, site: str, seed: int) -> pd.DataFrame:
    local_rng = np.random.default_rng(seed)
    rows = []
    object_low, object_high = ((-0.25, -0.20), (0.30, 0.28)) if not site.startswith("Site C") else ((-0.55, -0.38), (0.55, 0.40))
    ee_low, ee_high = ((-0.35, -0.32), (0.28, 0.30)) if not site.startswith("Site C") else ((-0.52, -0.40), (0.50, 0.40))
    for episode_id in range(n):
        obj = local_rng.uniform(object_low, object_high)
        tray = local_rng.uniform((0.18, 0.12), (0.42, 0.36))
        ee = local_rng.uniform(ee_low, ee_high)
        gripper = float(local_rng.choice([0.0, 1.0]))
        object_width_m = float(local_rng.uniform(0.025, 0.045) if not site.startswith("Site C") else local_rng.uniform(0.030, 0.065))
        action = expert_delta(ee, obj)
        rows.append({
            "episode_id": f"{site}-{episode_id:04d}", "site": site,
            "object_x": obj[0], "object_y": obj[1], "tray_x": tray[0], "tray_y": tray[1],
            "ee_x": ee[0], "ee_y": ee[1], "gripper_open": gripper,
            "object_width_m": object_width_m,
            "action_dx_m": action[0], "action_dy_m": action[1],
        })
    return pd.DataFrame(rows)


site_a = build_demonstrations(600, "Site A", SEED)
site_b = build_demonstrations(220, "Site B development only", SEED + 1)
site_c = build_demonstrations(220, "Site C reporting only; no model, feature, normalizer, threshold, chunk, or rule changes", SEED + 2)

VISUAL_FEATURES = ["object_x", "object_y", "tray_x", "tray_y"]
PROPRIO_FEATURES = VISUAL_FEATURES + ["ee_x", "ee_y", "gripper_open"]
TARGETS = ["action_dx_m", "action_dy_m"]

vision_only_policy = Ridge(alpha=0.01).fit(site_a[VISUAL_FEATURES], site_a[TARGETS])
proprioception_aware_policy = Ridge(alpha=0.01).fit(site_a[PROPRIO_FEATURES], site_a[TARGETS])

def vector_action_metrics(frame: pd.DataFrame, features: list[str], model: Ridge, label: str) -> dict:
    truth = frame[TARGETS].to_numpy()
    pred = model.predict(frame[features])
    error = np.linalg.norm(pred - truth, axis=1)
    return {
        "policy": label,
        "translation_vector_MAE_m": float(error.mean()),
        "translation_vector_RMSE_m": float(np.sqrt(np.mean(error ** 2))),
        "p95_vector_error_m": float(np.percentile(error, 95)),
    }


bc_metrics = pd.DataFrame([
    vector_action_metrics(site_b, VISUAL_FEATURES, vision_only_policy, "vision_only"),
    vector_action_metrics(site_b, PROPRIO_FEATURES, proprioception_aware_policy, "vision_plus_proprioception"),
])
assert bc_metrics.loc[bc_metrics.policy == "vision_plus_proprioception", "translation_vector_MAE_m"].item() < bc_metrics.loc[bc_metrics.policy == "vision_only", "translation_vector_MAE_m"].item()
bc_metrics


## 8. Low supervised action error does not equal closed-loop competence

We roll both policies forward toward one target. The action-aware model receives its current state each step. The visual-only policy cannot observe where its previous action moved the end effector, so its repeated command drifts. This is a deliberately controlled covariate-shift demonstration, not a robotics benchmark.


In [ ]:
def rollout_bc(model: Ridge, features: list[str], start: np.ndarray, target: np.ndarray, steps: int = 12) -> dict:
    ee = start.copy()
    distances = []
    support_distances = []
    train_states = site_a[["ee_x", "ee_y"]].to_numpy()
    for _ in range(steps):
        row = pd.DataFrame([{
            "object_x": target[0], "object_y": target[1], "tray_x": 0.35, "tray_y": 0.25,
            "ee_x": ee[0], "ee_y": ee[1], "gripper_open": 1.0,
        }])
        action = model.predict(row[features])[0]
        action_norm = np.linalg.norm(action)
        if action_norm > ARM_A.max_translation_delta_m:
            action = action / action_norm * ARM_A.max_translation_delta_m
        ee = ee + action
        distances.append(float(np.linalg.norm(target - ee)))
        support_distances.append(float(np.min(np.linalg.norm(train_states - ee, axis=1))))
    return {"final_distance_m": distances[-1], "success": distances[-1] <= 0.04, "distances": distances, "support": support_distances}


rollout_visual = rollout_bc(vision_only_policy, VISUAL_FEATURES, np.array([-0.34, -0.28]), np.array([0.28, 0.24]))
rollout_proprio = rollout_bc(proprioception_aware_policy, PROPRIO_FEATURES, np.array([-0.34, -0.28]), np.array([0.28, 0.24]))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(rollout_visual["distances"], marker="o", label="vision only")
ax.plot(rollout_proprio["distances"], marker="o", label="vision + proprioception")
ax.axhline(0.04, color="#16A3A5", linestyle="--", label="success radius")
ax.set(xlabel="autonomous step", ylabel="distance to target (m)", title="Teacher-forced action fit versus rollout feedback")
ax.legend()
plt.show()

pd.DataFrame([
    {"policy": "vision_only", "final_distance_m": rollout_visual["final_distance_m"], "task_success": rollout_visual["success"]},
    {"policy": "vision_plus_proprioception", "final_distance_m": rollout_proprio["final_distance_m"], "task_success": rollout_proprio["success"]},
])


## 9. A local VLA policy proxy proposes; it never authorizes

The next proxy combines a grounded goal with robot state and emits a short Cartesian-delta action. It exposes a familiar VLA-shaped interface while remaining deterministic and inspectable. It does not solve perception, grasp planning, inverse kinematics, contact, or controller dynamics.


In [ ]:
class LocalVLAPolicyProxy:
    engine = "local_vla_policy_proxy"
    foundation_model = False
    policy_version = "local-vla-policy-proxy/v1"

    def propose(self, observation: Observation, grounded: GroundedGoal, embodiment: EmbodimentContract,
                now_s: float, action_id: str = "proposal-1") -> ActionProposal | None:
        if grounded.state != "grounded":
            return None
        lookup = {obj.object_id: obj for obj in observation.objects}
        target = lookup[grounded.target_object_id]
        ee = np.asarray(observation.end_effector_position_m)
        displacement = np.asarray(target.position_m) - ee
        distance = np.linalg.norm(displacement)
        if distance <= 0.035:
            delta = np.zeros(2)
            skill, gripper = "grasp", "close"
        else:
            delta = expert_delta(ee, np.asarray(target.position_m), embodiment.max_translation_delta_m)
            skill, gripper = "move_to", "hold"
        return ActionProposal(
            action_id, observation.observation_id, observation.timestamp_s, observation.camera_captured_at_s, now_s,
            embodiment.robot_id, embodiment.contract_version, embodiment.control_mode,
            embodiment.action_frame, "metre", tuple(float(value) for value in delta), gripper, skill,
            target.object_id, self.policy_version, 0.94,
        )


LOCAL_POLICY = LocalVLAPolicyProxy()
observation = make_observation(unique_objects, now_s=12.4)
proposal = LOCAL_POLICY.propose(observation, unique_grounding, ARM_A, now_s=12.45)
assert proposal is not None and proposal.frame == "robot_base"
print(json.dumps(asdict(proposal), indent=2))


## 10. Deterministic reachability, collision, freshness, and skill checks

The restricted rectangle is a controlled collision proxy. Segment sampling is sufficient for this lesson's 2D geometry; it is not a swept-volume planner. We keep rejection reasons explicit and do not let policy confidence bypass them.


In [ ]:
RESTRICTED_RECT = {"xmin": 0.02, "xmax": 0.15, "ymin": -0.05, "ymax": 0.05}
MAX_OBSERVATION_AGE_S = 0.25  # selected on Site B; synthetic demonstration threshold

@dataclass(frozen=True)
class ValidationDecision:
    action_id: str
    allowed: bool
    reason_codes: tuple[str, ...]
    checked_at_s: float
    validator_version: str = "embodied-gateway/v1"


def point_reachable(point: np.ndarray, embodiment: EmbodimentContract) -> bool:
    return (
        embodiment.workspace_x_m[0] <= point[0] <= embodiment.workspace_x_m[1]
        and embodiment.workspace_y_m[0] <= point[1] <= embodiment.workspace_y_m[1]
    )


def segment_intersects_restricted(start: np.ndarray, end: np.ndarray, samples: int = 101) -> bool:
    points = np.linspace(start, end, samples)
    inside = (
        (points[:, 0] >= RESTRICTED_RECT["xmin"]) & (points[:, 0] <= RESTRICTED_RECT["xmax"])
        & (points[:, 1] >= RESTRICTED_RECT["ymin"]) & (points[:, 1] <= RESTRICTED_RECT["ymax"])
    )
    return bool(inside.any())


def validate_action(proposal: ActionProposal, observation: Observation, embodiment: EmbodimentContract,
                    grounded: GroundedGoal, now_s: float) -> ValidationDecision:
    reasons = []
    delta = np.asarray(proposal.translation_delta_m)
    start = np.asarray(observation.end_effector_position_m)
    end = start + delta
    if proposal.observation_id != observation.observation_id:
        reasons.append("observation_identity_mismatch")
    if (proposal.observation_timestamp_s != observation.timestamp_s
            or proposal.source_capture_timestamp_s != observation.camera_captured_at_s):
        reasons.append("observation_timestamp_mismatch")
    sensor_age_s = now_s - observation.camera_captured_at_s
    if sensor_age_s > MAX_OBSERVATION_AGE_S:
        reasons.append("stale_observation")
    if proposal.robot_id != embodiment.robot_id or proposal.embodiment_version != embodiment.contract_version:
        reasons.append("embodiment_mismatch")
    if (proposal.frame != embodiment.action_frame or proposal.control_mode != embodiment.control_mode
            or proposal.translation_unit != "metre"):
        reasons.append("action_contract_mismatch")
    if np.linalg.norm(delta) > embodiment.max_translation_delta_m + 1e-9:
        reasons.append("delta_limit_exceeded")
    if not point_reachable(end, embodiment):
        reasons.append("unreachable_target")
    if segment_intersects_restricted(start, end):
        reasons.append("collision_proxy_violation")
    if grounded.state != "grounded":
        reasons.append("goal_not_uniquely_grounded")
    if proposal.skill == "grasp":
        target = next((obj for obj in observation.objects if obj.object_id == proposal.target_object_id), None)
        if target is None:
            reasons.append("missing_grasp_target")
        elif not grasp_affordance(target, embodiment).compatible:
            reasons.append("gripper_incompatible")
        elif np.linalg.norm(np.asarray(target.position_m) - start) > 0.04:
            reasons.append("grasp_precondition_not_at_target")
    return ValidationDecision(proposal.action_id, not reasons, tuple(reasons), now_s)


decision = validate_action(proposal, observation, ARM_A, unique_grounding, now_s=12.46)
print(asdict(decision))


## 11. Policy confidence is not safety

This 0.99-confidence proposal crosses the restricted region. The deterministic gateway blocks it. Confidence calibration and geometric validity answer different questions.


In [ ]:
unsafe_high_confidence = ActionProposal(
    "unsafe-high-confidence", observation.observation_id, observation.timestamp_s, observation.camera_captured_at_s, 12.45,
    ARM_A.robot_id, ARM_A.contract_version, "cartesian_delta", "robot_base",
    "metre", (0.08, 0.04), "hold", "move_to", "red_block_1", "local-vla-policy-proxy/v1", 0.99,
)
unsafe_observation = replace(observation, end_effector_position_m=(-0.04, -0.02))
unsafe_high_confidence = replace(unsafe_high_confidence, observation_id=unsafe_observation.observation_id)
unsafe_decision = validate_action(unsafe_high_confidence, unsafe_observation, ARM_A, unique_grounding, now_s=12.46)
assert not unsafe_decision.allowed
assert "collision_proxy_violation" in unsafe_decision.reason_codes
pd.DataFrame([{**asdict(unsafe_high_confidence), **{"allowed": unsafe_decision.allowed, "reasons": unsafe_decision.reason_codes}}])


## 12. Freshness is part of action validity

The action binds to an exact observation timestamp. A delayed proposal is rejected and the required response is re-observation—not a confidence adjustment.


In [ ]:
stale_proposal = replace(proposal, action_id="stale-proposal", generated_at_s=13.0)
stale_decision = validate_action(stale_proposal, observation, ARM_A, unique_grounding, now_s=13.0)
freshness_examples = pd.DataFrame([
    {"case": "fresh", "sensor_age_s": 12.46 - observation.camera_captured_at_s,
     "policy_compute_age_s": proposal.generated_at_s - observation.timestamp_s,
     "total_action_latency_s": 12.46 - observation.camera_captured_at_s,
     "allowed": decision.allowed, "reasons": decision.reason_codes},
    {"case": "stale", "sensor_age_s": 13.0 - observation.camera_captured_at_s,
     "policy_compute_age_s": stale_proposal.generated_at_s - observation.timestamp_s,
     "total_action_latency_s": 13.0 - observation.camera_captured_at_s,
     "allowed": stale_decision.allowed, "reasons": stale_decision.reason_codes},
])
assert "stale_observation" in stale_decision.reason_codes
freshness_examples


## 13. Action chunking: open loop versus receding horizon

Both rollouts receive the same target displacement after step three. The open-loop policy consumes its original eight-step chunk. The receding-horizon policy executes one step, observes, and replans. This is the central feedback experiment.


In [ ]:
def plan_chunk(position: np.ndarray, target: np.ndarray, horizon: int = 8, step_m: float = 0.06) -> list[np.ndarray]:
    actions = []
    simulated = position.copy()
    for _ in range(horizon):
        delta = expert_delta(simulated, target, step_m)
        actions.append(delta)
        simulated = simulated + delta
    return actions


def disturbed_rollout(closed_loop: bool) -> dict:
    position = np.array([-0.28, -0.20])
    original_target = np.array([0.30, 0.18])
    shifted_target = original_target + np.array([-0.12, 0.14])
    target = original_target.copy()
    initial_chunk = plan_chunk(position, target)
    path = [position.copy()]
    for step in range(8):
        if step == 3:
            target = shifted_target.copy()
        delta = plan_chunk(position, target, horizon=1)[0] if closed_loop else initial_chunk[step]
        position = position + delta
        path.append(position.copy())
    path = np.asarray(path)
    return {
        "mode": "receding_horizon" if closed_loop else "open_loop_chunk",
        "final_distance_m": float(np.linalg.norm(position - shifted_target)),
        "task_success": bool(np.linalg.norm(position - shifted_target) <= 0.05),
        "path_length_m": float(np.linalg.norm(np.diff(path, axis=0), axis=1).sum()),
        "path": path,
        "target": shifted_target,
    }


open_loop_result = disturbed_rollout(False)
closed_loop_result = disturbed_rollout(True)
chunking_metrics = pd.DataFrame([{key: value for key, value in result.items() if key not in {"path", "target"}} for result in [open_loop_result, closed_loop_result]])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for result, style in [(open_loop_result, "o-"), (closed_loop_result, "s-")]:
    ax.plot(result["path"][:, 0], result["path"][:, 1], style, label=result["mode"])
ax.scatter(*closed_loop_result["target"], marker="*", s=180, color="#16A3A5", label="shifted target")
ax.set(xlabel="robot-base x (m)", ylabel="robot-base y (m)", title="Same disturbance, different feedback contract")
ax.axis("equal")
ax.legend()
plt.show()
chunking_metrics


## 14. Visual servoing and control-frequency budgets

A proportional visual-servo proxy converts current error into a bounded delta and re-observes after each correction. The timing table shows that 1, 10, and 50 Hz imply very different total deadlines. The numbers are synthetic demonstrations, not target-hardware measurements.


In [ ]:
def visual_servo(start: np.ndarray, target: np.ndarray, gain: float = 0.6, tolerance_m: float = 0.01, max_steps: int = 30) -> dict:
    position = start.copy()
    errors = []
    for step in range(max_steps):
        error = target - position
        errors.append(float(np.linalg.norm(error)))
        if errors[-1] <= tolerance_m:
            break
        delta = gain * error
        if np.linalg.norm(delta) > ARM_A.max_translation_delta_m:
            delta = delta / np.linalg.norm(delta) * ARM_A.max_translation_delta_m
        position += delta
    return {"steps": step, "final_error_m": errors[-1], "errors": errors}


servo_result = visual_servo(np.array([-0.20, -0.10]), np.array([0.22, 0.18]))
timing_components_ms = {"capture": 5.0, "sync_preprocess": 3.0, "policy": 38.0, "validation": 2.0, "dispatch": 2.0}
total_latency_ms = sum(timing_components_ms.values())
frequency_table = pd.DataFrame([
    {
        "control_frequency_hz": hz,
        "deadline_ms": 1000.0 / hz,
        "synthetic_pipeline_ms": total_latency_ms,
        "deadline_met": total_latency_ms <= 1000.0 / hz,
        "remaining_budget_ms": 1000.0 / hz - total_latency_ms,
    }
    for hz in [1, 10, 50]
])
print({"visual_servo": {"steps": servo_result["steps"], "final_error_m": servo_result["final_error_m"]}})
frequency_table


## 15. A digest-bound, expiring, single-use simulation permit

Validation is advisory. The trusted notebook application issues a separate permit only after validation. The permit binds the exact proposal digest, local environment, expiry, and nonce. The executor rejects mutation, replay, expiry, and environment mismatch. This is an authorization-pattern lesson—not a physical safety certification.


In [ ]:
def canonical_digest(value: object) -> str:
    payload = asdict(value) if hasattr(value, "__dataclass_fields__") else value
    return sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


@dataclass(frozen=True)
class SimulationPermit:
    permit_id: str
    action_id: str
    action_digest: str
    environment_id: Literal["local_workcell_v1"]
    issued_at_s: float
    expires_at_s: float
    nonce: str
    scope: Literal["simulation_only"] = "simulation_only"


def issue_simulation_permit(proposal: ActionProposal, decision: ValidationDecision, now_s: float) -> SimulationPermit:
    assert decision.allowed and decision.action_id == proposal.action_id
    digest = canonical_digest(proposal)
    return SimulationPermit(
        permit_id=f"permit-{digest[:10]}", action_id=proposal.action_id, action_digest=digest,
        environment_id="local_workcell_v1", issued_at_s=now_s, expires_at_s=now_s + 0.10,
        nonce=sha256(f"{proposal.action_id}:{now_s}:{SEED}".encode()).hexdigest()[:16],
    )


class SimulationExecutor:
    def __init__(self) -> None:
        self.consumed_nonces: set[str] = set()

    def execute(self, proposal: ActionProposal, permit: SimulationPermit, now_s: float, environment_id: str,
                start_position: tuple[float, float]) -> tuple[float, float]:
        assert permit.scope == "simulation_only"
        assert environment_id == permit.environment_id
        assert now_s <= permit.expires_at_s
        assert permit.nonce not in self.consumed_nonces
        assert permit.action_id == proposal.action_id
        assert permit.action_digest == canonical_digest(proposal)
        self.consumed_nonces.add(permit.nonce)
        return tuple(np.asarray(start_position) + np.asarray(proposal.translation_delta_m))


safe_observation = replace(observation, end_effector_position_m=(-0.20, -0.20))
safe_proposal = replace(proposal, action_id="safe-proposal", observation_id=safe_observation.observation_id,
                        translation_delta_m=(0.04, 0.04), generated_at_s=12.45)
safe_decision = validate_action(safe_proposal, safe_observation, ARM_A, unique_grounding, now_s=12.46)
assert safe_decision.allowed
permit = issue_simulation_permit(safe_proposal, safe_decision, now_s=12.46)
executor = SimulationExecutor()
simulated_position = executor.execute(safe_proposal, permit, 12.47, "local_workcell_v1", safe_observation.end_effector_position_m)

replay_blocked = False
try:
    executor.execute(safe_proposal, permit, 12.48, "local_workcell_v1", safe_observation.end_effector_position_m)
except AssertionError:
    replay_blocked = True
assert replay_blocked

mutated_blocked = False
fresh_executor = SimulationExecutor()
try:
    fresh_executor.execute(replace(safe_proposal, translation_delta_m=(0.05, 0.04)), permit, 12.47, "local_workcell_v1", safe_observation.end_effector_position_m)
except AssertionError:
    mutated_blocked = True
assert mutated_blocked

print({"simulated_position": simulated_position, "replay_blocked": replay_blocked, "mutated_proposal_blocked": mutated_blocked})


## 16. Skill preconditions, postconditions, and recovery

The `grasp` preconditions include unique grounding, compatibility, proximity, freshness, and feasibility. Postcondition verification uses measured object motion and gripper state. Issuing `close` is not success. The failure case triggers one bounded re-observe-and-retry recovery, then stops for review.


In [ ]:
@dataclass(frozen=True)
class PostconditionEvidence:
    action_id: str
    skill: Literal["grasp", "release"]
    source_observation_id: str
    result_observation_id: str
    object_displacement_m: float
    gripper_state_consistent: bool
    destination_error_m: float | None
    task_effect_verified: bool
    next_state: Literal["continue", "recover", "review_required"]


def verify_grasp(action_id: str, source_observation_id: str, result_observation_id: str,
                 object_before: np.ndarray, object_after: np.ndarray,
                 gripper_width_after_m: float, object_width_m: float) -> PostconditionEvidence:
    displacement_m = float(np.linalg.norm(object_after - object_before))
    moved = displacement_m >= 0.015
    closed_on_object = object_width_m * 0.7 <= gripper_width_after_m <= object_width_m * 1.3
    verified = bool(moved and closed_on_object)
    return PostconditionEvidence(
        action_id, "grasp", source_observation_id, result_observation_id, displacement_m,
        bool(closed_on_object), None, verified, "continue" if verified else "recover",
    )


def verify_release(action_id: str, source_observation_id: str, result_observation_id: str,
                   object_before: np.ndarray, object_after: np.ndarray, destination: np.ndarray,
                   gripper_width_after_m: float, object_width_m: float, tolerance_m: float = 0.03) -> PostconditionEvidence:
    displacement_m = float(np.linalg.norm(object_after - object_before))
    destination_error_m = float(np.linalg.norm(object_after - destination))
    gripper_open = gripper_width_after_m >= object_width_m * 1.5
    verified = bool(destination_error_m <= tolerance_m and gripper_open)
    return PostconditionEvidence(
        action_id, "release", source_observation_id, result_observation_id, displacement_m,
        bool(gripper_open), destination_error_m, verified, "continue" if verified else "recover",
    )


failed_grasp = verify_grasp("grasp-1", "obs-before-1", "obs-after-1", np.array([0.12, 0.12]), np.array([0.122, 0.12]), 0.0, 0.04)
recovered_grasp = verify_grasp("grasp-1-retry", "obs-before-2", "obs-after-2", np.array([0.12, 0.12]), np.array([0.15, 0.13]), 0.038, 0.04)
verified_release = verify_release("release-1", "obs-before-3", "obs-after-3", np.array([0.34, 0.23]), np.array([0.36, 0.24]), np.array([0.36, 0.24]), 0.08, 0.04)
postcondition_table = pd.DataFrame([asdict(failed_grasp), asdict(recovered_grasp), asdict(verified_release)])
recovery_trials = pd.DataFrame([{
    "eligible_failure_action_id": failed_grasp.action_id, "attempted": True,
    "recovery_action_id": recovered_grasp.action_id, "succeeded": recovered_grasp.task_effect_verified,
}])
recovery_attempt_rate = float(recovery_trials["attempted"].mean())
recovery_success_rate = float(recovery_trials.loc[recovery_trials.attempted, "succeeded"].mean())
assert failed_grasp.next_state == "recover" and recovered_grasp.task_effect_verified
assert verified_release.task_effect_verified
postcondition_table


### End-to-end closed-loop pick-and-place episode

The next episode actually completes the stated red-block-to-blue-tray goal. Every step rebuilds a timestamped observation, re-grounds the goal, proposes one bounded prefix, validates it, obtains a new single-use simulation permit, executes locally, and captures a new observation. Grasp and release update controller state only after `verify_grasp` or `verify_release` accepts source-bound postcondition evidence. The deterministic simulator makes successful effects observable; it does not let the requested skill self-certify success.


In [ ]:
def propose_pick_place_step(observation: Observation, grounded: GroundedGoal, held: bool, now_s: float, step: int) -> ActionProposal:
    lookup = {obj.object_id: obj for obj in observation.objects}
    target_id = grounded.destination_object_id if held else grounded.target_object_id
    target = np.asarray(lookup[target_id].position_m)
    ee = np.asarray(observation.end_effector_position_m)
    distance = np.linalg.norm(target - ee)
    if distance <= 0.035:
        skill, gripper, delta = ("release", "open", np.zeros(2)) if held else ("grasp", "close", np.zeros(2))
    else:
        skill, gripper, delta = "move_to", "hold", expert_delta(ee, target, ARM_A.max_translation_delta_m)
    return ActionProposal(
        f"episode-action-{step:02d}", observation.observation_id, observation.timestamp_s, observation.camera_captured_at_s, now_s,
        ARM_A.robot_id, ARM_A.contract_version, ARM_A.control_mode, ARM_A.action_frame,
        "metre", tuple(float(value) for value in delta), gripper, skill, target_id,
        "closed-loop-pick-place-proxy/v1", 0.93,
    )


ee = np.array([-0.20, 0.20])
block = np.array([0.12, 0.12])
tray = np.array([0.36, 0.24])
held = False
task_success = False
episode_trace = []
episode_postconditions = []
episode_executor = SimulationExecutor()
for step in range(20):
    now_s = 20.0 + step * 0.10
    objects = [
        SceneObject("red_block_1", "block", "red", tuple(block), "robot_base", 0.04),
        SceneObject("blue_tray_1", "tray", "blue", tuple(tray), "robot_base", 0.16),
        SceneObject("restricted_1", "obstacle", "gray", (0.08, 0.0), "robot_base", 0.12),
    ]
    episode_observation = make_observation(objects, source_id="Site B closed loop", now_s=now_s, ee=tuple(ee),
                                           gripper_width_m=0.04 if held else 0.08)
    episode_grounding = LocalGroundingProxy.ground(GOAL, episode_observation)
    episode_proposal = propose_pick_place_step(episode_observation, episode_grounding, held, now_s + 0.02, step)
    episode_decision = validate_action(episode_proposal, episode_observation, ARM_A, episode_grounding, now_s + 0.03)
    assert episode_decision.allowed, episode_decision.reason_codes
    episode_permit = issue_simulation_permit(episode_proposal, episode_decision, now_s + 0.03)
    executed_ee = np.asarray(episode_executor.execute(episode_proposal, episode_permit, now_s + 0.04,
                                                      "local_workcell_v1", tuple(ee)))
    transition = "motion"
    postcondition_verified = None
    observed_ee = executed_ee.copy()
    observed_block = executed_ee.copy() if held else block.copy()
    observed_gripper_width_m = 0.04 if held else 0.08
    if episode_proposal.skill == "grasp":
        # The simulated grasp skill closes and performs a small diagnostic lift before re-observation.
        observed_ee = executed_ee + np.array([0.0, 0.02])
        observed_block = observed_ee.copy()
        observed_gripper_width_m = 0.04
    elif episode_proposal.skill == "release":
        observed_block = tray.copy()
        observed_gripper_width_m = 0.08
    result_objects = [
        SceneObject("red_block_1", "block", "red", tuple(observed_block), "robot_base", 0.04),
        SceneObject("blue_tray_1", "tray", "blue", tuple(tray), "robot_base", 0.16),
        SceneObject("restricted_1", "obstacle", "gray", (0.08, 0.0), "robot_base", 0.12),
    ]
    result_observation = make_observation(
        result_objects, source_id="Site B postcondition", now_s=now_s + 0.05,
        ee=tuple(observed_ee), gripper_width_m=observed_gripper_width_m,
    )
    source_block_observed = next(obj for obj in episode_observation.objects if obj.object_id == "red_block_1")
    result_block_observed = next(obj for obj in result_observation.objects if obj.object_id == "red_block_1")
    result_tray_observed = next(obj for obj in result_observation.objects if obj.object_id == "blue_tray_1")
    if episode_proposal.skill == "grasp":
        post = verify_grasp(
            episode_proposal.action_id, episode_observation.observation_id, result_observation.observation_id,
            np.asarray(source_block_observed.position_m), np.asarray(result_block_observed.position_m),
            result_observation.gripper_width_m, source_block_observed.width_m,
        )
        episode_postconditions.append(asdict(post))
        postcondition_verified = post.task_effect_verified
        transition = "grasp_verified" if post.task_effect_verified else "grasp_recovery_required"
        if post.task_effect_verified:
            held, ee, block = True, observed_ee, observed_block
    elif episode_proposal.skill == "release":
        post = verify_release(
            episode_proposal.action_id, episode_observation.observation_id, result_observation.observation_id,
            np.asarray(source_block_observed.position_m), np.asarray(result_block_observed.position_m),
            np.asarray(result_tray_observed.position_m), result_observation.gripper_width_m, source_block_observed.width_m,
        )
        episode_postconditions.append(asdict(post))
        postcondition_verified = post.task_effect_verified
        transition = "release_verified" if post.task_effect_verified else "release_recovery_required"
        if post.task_effect_verified:
            held, ee, block, task_success = False, observed_ee, observed_block, True
    else:
        ee, block = observed_ee, observed_block
    episode_trace.append({
        "step": step, "skill": episode_proposal.skill, "allowed": episode_decision.allowed,
        "transition": transition, "result_observation_id": result_observation.observation_id,
        "postcondition_verified": postcondition_verified,
        "ee_x": ee[0], "ee_y": ee[1], "held_after": held,
    })
    if task_success:
        break

closed_loop_task_metrics = {
    "task_success": task_success, "steps_to_completion": len(episode_trace),
    "verified_grasp_count": sum(row["skill"] == "grasp" and row["task_effect_verified"] for row in episode_postconditions),
    "verified_release_count": sum(row["skill"] == "release" and row["task_effect_verified"] for row in episode_postconditions),
    "executed_constraint_violations": 0, "physical_authorization": "none",
}
assert task_success and np.allclose(block, tray)
assert closed_loop_task_metrics["verified_grasp_count"] == 1
assert closed_loop_task_metrics["verified_release_count"] == 1
pd.DataFrame(episode_trace)


## 17. Proposal violations versus executed violations

Blocking unsafe proposals is useful evidence only when valid work is also measured. This suite includes valid, stale, capture-timestamp-mismatch, wrong-embodiment, oversized-step, unreachable, collision, ambiguous-goal, and incompatible-gripper cases. Nothing invalid reaches the simulation executor.


In [ ]:
validation_cases = []

def record_case(name: str, candidate: ActionProposal, obs: Observation, embodiment: EmbodimentContract,
                grounded: GroundedGoal, now_s: float, expected_valid: bool) -> None:
    outcome = validate_action(candidate, obs, embodiment, grounded, now_s)
    validation_cases.append({
        "case": name, "expected_valid": expected_valid, "allowed": outcome.allowed,
        "reason_codes": outcome.reason_codes, "policy_confidence": candidate.policy_confidence,
    })


record_case("valid", safe_proposal, safe_observation, ARM_A, unique_grounding, 12.46, True)
record_case("stale", replace(safe_proposal, action_id="case-stale"), safe_observation, ARM_A, unique_grounding, 13.0, False)
record_case("capture_timestamp_mismatch", replace(safe_proposal, action_id="case-capture", source_capture_timestamp_s=12.39), safe_observation, ARM_A, unique_grounding, 12.46, False)
record_case("wrong_embodiment", replace(safe_proposal, action_id="case-emb", robot_id="arm_B"), safe_observation, ARM_A, unique_grounding, 12.46, False)
record_case("oversized", replace(safe_proposal, action_id="case-large", translation_delta_m=(0.20, 0.0)), safe_observation, ARM_A, unique_grounding, 12.46, False)
edge_observation = replace(safe_observation, end_effector_position_m=(0.58, 0.45))
record_case("unreachable", replace(safe_proposal, action_id="case-reach", observation_id=edge_observation.observation_id,
                                    translation_delta_m=(0.08, 0.08)), edge_observation, ARM_A, unique_grounding, 12.46, False)
record_case("collision", unsafe_high_confidence, unsafe_observation, ARM_A, unique_grounding, 12.46, False)
record_case("ambiguous", replace(safe_proposal, action_id="case-ambiguous"), safe_observation, ARM_A, ambiguous_grounding, 12.46, False)

narrow_observation = replace(safe_observation, embodiment_version=ARM_B.contract_version)
wide_grasp = ActionProposal(
    "case-gripper", narrow_observation.observation_id, narrow_observation.timestamp_s, narrow_observation.camera_captured_at_s, 12.45,
    ARM_B.robot_id, ARM_B.contract_version, "cartesian_delta", "robot_base", "metre", (0.0, 0.0),
    "close", "grasp", "red_block_1", "local-vla-policy-proxy/v1", 0.97,
)
wide_target = replace(wide_block, object_id="red_block_1")
wide_scene = tuple(wide_target if obj.object_id == "red_block_1" else obj for obj in narrow_observation.objects)
narrow_observation = replace(narrow_observation, objects=wide_scene, end_effector_position_m=wide_target.position_m)
record_case("incompatible_gripper", wide_grasp, narrow_observation, ARM_B, unique_grounding, 12.46, False)

validation_table = pd.DataFrame(validation_cases)
proposal_violation_rate = float((~validation_table.expected_valid).mean())
executed_violation_rate = float((~validation_table.loc[validation_table.allowed, "expected_valid"]).mean())
valid_work_block_rate = float((~validation_table.loc[validation_table.expected_valid, "allowed"]).mean())
assert validation_table.allowed.tolist() == validation_table.expected_valid.tolist()
assert "observation_timestamp_mismatch" in validation_table.loc[validation_table.case == "capture_timestamp_mismatch", "reason_codes"].item()
assert "gripper_incompatible" in validation_table.loc[validation_table.case == "incompatible_gripper", "reason_codes"].item()
assert executed_violation_rate == 0.0
validation_table


## 18. Freeze on Site B, then report Site C embodiment shift

The complete selected policy—including feature set, fitted coefficients, normalizer, freshness threshold, chunk policy, and validator version—is hashed before Site C is inspected. Site C broadens workspace and object-width support, so the same frozen proposals encounter both ARM B reach limits and objects that cross its gripper-capacity boundary. The report separates reachability, affordance compatibility, and their combined feasible-action rate for ARM A versus ARM B without tuning.


In [ ]:
def frozen_policy_payload() -> dict:
    return {
        "model": "Ridge",
        "features": PROPRIO_FEATURES,
        "coef": proprioception_aware_policy.coef_.round(12).tolist(),
        "intercept": proprioception_aware_policy.intercept_.round(12).tolist(),
        "normalizer": asdict(NORMALIZER),
        "freshness_threshold_s": MAX_OBSERVATION_AGE_S,
        "chunk_execution": "receding_horizon_prefix_1",
        "validator_version": "embodied-gateway/v1",
        "development_source": "Site B development only",
    }


FROZEN_POLICY_HASH = canonical_digest(frozen_policy_payload())
policy_hash_before_site_c = FROZEN_POLICY_HASH

site_c_action_metrics = vector_action_metrics(site_c, PROPRIO_FEATURES, proprioception_aware_policy, "frozen_site_c")
site_c_pred = proprioception_aware_policy.predict(site_c[PROPRIO_FEATURES])
site_c_end = site_c[["ee_x", "ee_y"]].to_numpy() + site_c_pred
site_c_reachable_arm_a = np.array([point_reachable(point, ARM_A) for point in site_c_end])
site_c_reachable_arm_b = np.array([point_reachable(point, ARM_B) for point in site_c_end])
site_c_affordance_arm_a = site_c["object_width_m"].to_numpy() + 0.01 <= ARM_A.gripper_max_width_m
site_c_affordance_arm_b = site_c["object_width_m"].to_numpy() + 0.01 <= ARM_B.gripper_max_width_m
site_c_embodiment_report = {
    **site_c_action_metrics,
    "candidate_reachable_rate_arm_A": float(site_c_reachable_arm_a.mean()),
    "candidate_reachable_rate_arm_B": float(site_c_reachable_arm_b.mean()),
    "affordance_compatible_rate_arm_A": float(site_c_affordance_arm_a.mean()),
    "affordance_compatible_rate_arm_B": float(site_c_affordance_arm_b.mean()),
    "combined_feasible_action_rate_arm_A": float((site_c_reachable_arm_a & site_c_affordance_arm_a).mean()),
    "combined_feasible_action_rate_arm_B": float((site_c_reachable_arm_b & site_c_affordance_arm_b).mean()),
    "reporting_only_no_changes": True,
}

policy_hash_after_site_c = canonical_digest(frozen_policy_payload())
assert policy_hash_before_site_c == policy_hash_after_site_c
assert site_c_embodiment_report["candidate_reachable_rate_arm_B"] < site_c_embodiment_report["candidate_reachable_rate_arm_A"]
assert site_c_embodiment_report["affordance_compatible_rate_arm_B"] < site_c_embodiment_report["affordance_compatible_rate_arm_A"]
assert site_c_embodiment_report["combined_feasible_action_rate_arm_B"] < site_c_embodiment_report["combined_feasible_action_rate_arm_A"]
pd.DataFrame([site_c_embodiment_report])


## 19. Failure attribution and evaluation matrix

Task success is an end-to-end metric; it does not locate the fault. The earliest observable failure taxonomy below keeps observation, grounding, affordance, action representation, policy, feasibility, authorization, execution, and verification distinct.


In [ ]:
failure_attribution = pd.DataFrame([
    ["stale", "observation", "freshness gate rejected", "re-observe"],
    ["ambiguous", "grounding", "two target candidates", "clarification_required"],
    ["incompatible_gripper", "affordance", "tool aperture too small", "change tool or stop"],
    ["wrong_frame", "action_representation", "frame/contract mismatch", "reject schema"],
    ["vision_only_drift", "policy", "missing proprioception", "restore state input"],
    ["collision", "feasibility", "restricted region intersected", "replan"],
    ["replay", "authorization", "permit nonce consumed", "reject"],
    ["failed_grasp", "verification", "postcondition absent", "bounded recovery"],
], columns=["case", "earliest_failure_layer", "evidence", "response"])

evaluation_matrix = pd.DataFrame([
    ["grounding", "unambiguous_binding_accuracy", 1.0],
    ["grounding", "ambiguity_clarification_recall", 1.0],
    ["grounding", "unsafe_action_after_ambiguity_rate", 0.0],
    ["action", "site_b_translation_vector_MAE_m", bc_metrics.loc[bc_metrics.policy == "vision_plus_proprioception", "translation_vector_MAE_m"].item()],
    ["action", "tokenization_translation_vector_error_m", tokenization_translation_vector_error_m],
    ["rollout", "open_loop_disturbed_success", float(open_loop_result["task_success"])],
    ["rollout", "receding_horizon_disturbed_success", float(closed_loop_result["task_success"])],
    ["constraints", "proposal_violation_rate", proposal_violation_rate],
    ["constraints", "executed_violation_rate", executed_violation_rate],
    ["constraints", "valid_work_block_rate", valid_work_block_rate],
    ["feedback", "postcondition_recovery_success", float(recovered_grasp.task_effect_verified)],
    ["feedback", "recovery_attempt_rate", recovery_attempt_rate],
    ["feedback", "recovery_success_rate", recovery_success_rate],
    ["feedback", "episode_verified_grasp_count", float(closed_loop_task_metrics["verified_grasp_count"])],
    ["feedback", "episode_verified_release_count", float(closed_loop_task_metrics["verified_release_count"])],
    ["task", "closed_loop_pick_place_success", float(closed_loop_task_metrics["task_success"])],
    ["task", "steps_to_completion", float(closed_loop_task_metrics["steps_to_completion"])],
    ["timing", "stale_case_blocked", float(not stale_decision.allowed)],
], columns=["layer", "metric", "value"])
evaluation_matrix


## 20. Common SDKs and optional-system governance

The runnable course uses standard Python, NumPy, pandas, Matplotlib, and scikit-learn APIs. Heavy robot frameworks and VLA checkpoints are disabled, revision-pinned mappings only. A repository license does not automatically cover model weights, training data, robot assets, or transitive code. Enabling any adapter would require an isolated environment, exact artifact IDs, preprocessing/action-statistics review, target-embodiment evaluation, and a new safety case.


In [ ]:
OPTIONAL_TOOL_MANIFESTS = {
    "LeRobot": {"enabled": False, "revision": "5aa74557f84c54d4b458f8b9643c5aa2982acfed", "code_license": "Apache-2.0"},
    "OpenVLA": {"enabled": False, "revision": "c8f03f48af692657d3060c19588038c7220e9af9", "code_license": "MIT"},
    "openpi": {"enabled": False, "revision": "215abfb217dbac7d5f1273282331b9b1866c0479", "code_license": "Apache-2.0"},
    "Isaac_GR00T": {"enabled": False, "revision": "51d4c89f72fda44cbf77285c6a8114b52676b8a1", "code_license": "Apache-2.0"},
    "ManiSkill": {"enabled": False, "revision": "62ff3a5896b4d5b4cf0ac4c8d79afe600c9404a3", "code_license": "Apache-2.0"},
    "MuJoCo": {"enabled": False, "revision": "0452d71b0d6171551e2f0b06a87df925244f37d4", "code_license": "Apache-2.0"},
    "robomimic": {"enabled": False, "revision": "d309eaecc18acf4152a830a895a6984b8ac71b05", "code_license": "MIT"},
}
CV_ENABLE_LEROBOT=False
CV_ENABLE_OPENVLA=False
CV_ENABLE_OPENPI=False
CV_ENABLE_GR00T=False
CV_ENABLE_MANISKILL=False
CV_ENABLE_MUJOCO=False

assert not any(item["enabled"] for item in OPTIONAL_TOOL_MANIFESTS.values())
pd.DataFrame(OPTIONAL_TOOL_MANIFESTS).T


## 21. Export auditable, non-authorizing evidence

The artifact records measured local evidence, disabled optional mappings, evaluation boundaries, reason-coded failures, and unresolved production assumptions. It does not claim physical safety, current model-family superiority, or benchmark reproduction.


In [ ]:
evidence = {
    "course": "Advanced 04 — Embodied Vision & Vision-Language-Action Models",
    "scenario": "synthetic_2d_workcell",
    "runtime_boundary": {
        "engine": "local_simulation_proxy",
        "foundation_model": False,
        "robot_connected": False,
        "physical_authorization": "none",
        "simulation_permit_scope": "local_workcell_v1_only",
    },
    "contracts": {
        "embodiment": asdict(ARM_A),
        "held_out_embodiment": asdict(ARM_B),
        "normalizer": asdict(NORMALIZER),
        "action_frame_required": True,
        "translation_unit": "metre",
        "freshness_threshold_s": MAX_OBSERVATION_AGE_S,
        "freshness_anchor": "camera_captured_at_s",
        "threshold_notice": DEMONSTRATION_THRESHOLD_NOTICE,
    },
    "evaluation_data_boundary": {
        "Site_A": "training demonstrations",
        "Site_B": "development only",
        "Site_C": "reporting only; no changes",
        "policy_hash_before_site_c": policy_hash_before_site_c,
        "policy_hash_after_site_c": policy_hash_after_site_c,
    },
    "locally_measured_evidence": {
        "grounding_cases": grounding_cases.to_dict(orient="records"),
        "affordance_by_embodiment": affordance_table.to_dict(orient="records"),
        "behavioral_cloning": bc_metrics.to_dict(orient="records"),
        "action_tokenization_translation_error_m": tokenization_translation_vector_error_m,
        "chunking": chunking_metrics.to_dict(orient="records"),
        "freshness": freshness_examples.to_dict(orient="records"),
        "validation_cases": validation_table.to_dict(orient="records"),
        "postconditions": postcondition_table.to_dict(orient="records"),
        "recovery_trials": recovery_trials.to_dict(orient="records"),
        "closed_loop_task": closed_loop_task_metrics,
        "closed_loop_trace": episode_trace,
        "closed_loop_postconditions": episode_postconditions,
        "site_c": site_c_embodiment_report,
        "evaluation_matrix": evaluation_matrix.to_dict(orient="records"),
    },
    "optional_model_observations": [],
    "optional_tool_manifests": OPTIONAL_TOOL_MANIFESTS,
    "failure_attribution": failure_attribution.to_dict(orient="records"),
    "unresolved_production_assumptions": [
        "3D calibration and uncertainty", "inverse kinematics and dynamics", "swept-volume collision",
        "contact and grasp stability", "hardware timing and watchdogs", "safety-rated stop path",
        "dataset/model/checkpoint rights", "human factors and incident response", "sim-to-real evidence",
    ],
    "authorization": "none",
}

(ARTIFACT_DIR / "embodied_vla_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
decision_rows = pd.DataFrame([
    {"decision": "continue_in_local_simulation", "allowed": True, "scope": "local_workcell_v1", "reason": "all deterministic checks passed"},
    {"decision": "connect_physical_robot", "allowed": False, "scope": "none", "reason": "not evaluated or authorized"},
    {"decision": "report_site_c", "allowed": True, "scope": "frozen_policy_evidence", "reason": "policy hash unchanged"},
])
decision_rows.to_csv(ARTIFACT_DIR / "embodied_vla_decision.csv", index=False)

assert evidence["runtime_boundary"]["physical_authorization"] == "none"
assert policy_hash_before_site_c == policy_hash_after_site_c
assert evaluation_matrix.query("metric == 'executed_violation_rate'")["value"].item() == 0.0
print("Advanced 04 invariant checks passed; artifacts:", ARTIFACT_DIR)


## 22. What you should now be able to explain without code

1. Why does a correct visual description not imply a physically valid action?
2. Why can the same instruction require different actions for two embodiments?
3. Why is an action vector invalid without frame, units, controller, and timing?
4. Why should two matching referents cause clarification rather than a high-confidence guess?
5. How does an affordance differ from an object class and from action feasibility?
6. Why can low action-prediction loss coexist with poor rollout success?
7. What do action chunks gain, and what feedback do they postpone?
8. Why can high policy confidence coexist with a collision violation?
9. What is the difference between validation, a scoped permit, execution, and verified success?
10. What evidence is still missing after a successful simulation?

**Next:** Advanced 05 adds persistent spatial relations, scene graphs, navigation state, and queryable spatial memory to the verified embodied loop.
